# ── 0. 配置区 ──────────────────────────────────────────────

In [1]:
from pathlib import Path

# GT 和预测 COCO JSON 路径
GT_PATH   = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/annotations/coco_detection.json")
PRED_PATH = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/01_all_labels_nocls.json")

# 评估参数
IOU_THR   = 0.9
SCORE_THR = 0.5

# 输出路径
OUT_CSV = Path("/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp") / f"per_image_eval_iou{int(IOU_THR*100)}_score_{SCORE_THR}_by_filename.csv"

# ── 1. 初始化日志 ──────────────────────────────────────────

In [2]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

2026-02-19 17:12:12,816 [INFO] ============ Notebook logging 初始化完成 ============


# ── 2. Step 1：运行 per-image GT vs 预测评估 ────────────────

In [3]:
# ── Step 1：运行评估，计算 TP/FP/FN/Precision/Recall/F1 ──────
# 输入：GT_PATH（COCO JSON）、PRED_PATH（COCO JSON）
# 输出：OUT_CSV（per-image 评估结果）

logger.info("Step 1 开始：per-image GT vs 预测评估")
logger.info(f"  GT:   {GT_PATH}")
logger.info(f"  Pred: {PRED_PATH}")
logger.info(f"  IoU={IOU_THR}, ScoreThr={SCORE_THR}")

import pandas as pd
from ty_run_tools import per_image_eval_by_filename

try:
    df = per_image_eval_by_filename(
        gt_path=GT_PATH,
        pred_path=PRED_PATH,
        iou_thr=IOU_THR,
        score_thr=SCORE_THR,
    )
    averages = df[['gt_count', 'pred_count', 'tp (Hit)', 'fp (Error Warning)', 'fn (Miss)', 'precision', 'recall', 'f1']].mean()
    averages['file_name'] = 'Average'
    df = pd.concat([df, pd.DataFrame([averages])], ignore_index=True)
    df.to_csv(OUT_CSV, index=False)
    logger.info(f"Step 1 完成：共 {len(df)-1} 张图像，结果保存至 {OUT_CSV}")
except Exception as e:
    logger.error(f"评估失败: {e}")
    raise

2026-02-19 17:12:12,844 [INFO] Step 1 开始：per-image GT vs 预测评估
2026-02-19 17:12:12,844 [INFO]   GT:   /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/annotations/coco_detection.json
2026-02-19 17:12:12,845 [INFO]   Pred: /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/01_all_labels_nocls.json
2026-02-19 17:12:12,845 [INFO]   IoU=0.9, ScoreThr=0.5
2026-02-19 17:12:13,723 [INFO] Step 1 完成：共 36 张图像，结果保存至 /home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/00_pipeline/00_temp/per_image_eval_iou90_score_0.5_by_filename.csv


# ── 3. 验证 ────────────────────────────────────────────────

In [4]:
# ── 验证：展示评估结果 ────────────────────────────────────────
from IPython.display import display

logger.info(f"验证：df.shape={df.shape}，最后一行（Average）:")
display(df)

2026-02-19 17:12:13,727 [INFO] 验证：df.shape=(37, 9)，最后一行（Average）:


,file_name,gt_count,pred_count,tp (Hit),fp (Error Warning),fn (Miss),precision,recall,f1
0,0717_2003_580.jpg,0.000000,3.0,0.000000,3.000000,0.000000,0.000000,0.000000,0.000000
1,0717_2033_580.jpg,1.000000,4.0,1.000000,3.000000,0.000000,0.250000,1.000000,0.400000
2,0718_0604_580.jpg,113.000000,103.0,103.000000,0.000000,10.000000,1.000000,0.911504,0.953704
3,0718_0634_580.jpg,89.000000,81.0,80.000000,1.000000,9.000000,0.987654,0.898876,0.941176
4,0718_0704_580.jpg,82.000000,70.0,70.000000,0.000000,12.000000,1.000000,0.853659,0.921053
5,0718_0734_580.jpg,69.000000,59.0,56.000000,3.000000,13.000000,0.949153,0.811594,0.875000
6,0718_0804_580.jpg,60.000000,43.0,42.000000,1.000000,18.000000,0.976744,0.700000,0.815534
7,0718_0834_580.jpg,65.000000,47.0,47.000000,0.000000,18.000000,1.000000,0.723077,0.839286
8,0718_0904_580.jpg,54.000000,45.0,45.000000,0.000000,9.000000,1.000000,0.833333,0.909091
9,0718_0934_580.jpg,56.000000,48.0,48.000000,0.000000,8.000000,1.000000,0.857143,0.923077


In [6]:
import dtale
dtale.show(df).open_browser()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# ====== 1) 准备数据 ======
chart_data = df[['gt_count', 'pred_count']].copy()
chart_data = chart_data.dropna()
chart_data = chart_data.sort_values('gt_count')
chart_data = chart_data.rename(columns={'gt_count': 'x'})

# ====== 2) 先用 px 生成 OLS trendline 并取 R2 ======
fig_px = px.scatter(
    chart_data,
    x='x',
    y='pred_count',
    trendline='ols',
    trendline_color_override='red',
)

# 取拟合结果（statsmodels）
res_df = px.get_trendline_results(fig_px)  # DataFrame
# 如果没有 color/ facet 分组，一般就一行
model = res_df.loc[0, "px_fit_results"]
r2 = float(model.rsquared)

# trendline trace（px 里通常第 2 条 trace 是 trendline）
trendline_trace = fig_px.data[1]

# ====== 3) 用 go 画散点 + trendline，并把 R2 展示出来 ======
scatter = go.Scattergl(
    x=chart_data['x'],
    y=chart_data['pred_count'],
    mode='markers',
    opacity=0.7,
    name='all',
    marker=dict(size=15, line=dict(width=0.5, color='white')),
)

fig = go.Figure(
    data=[scatter, trendline_trace],
    layout=go.Layout(
        width=900,
        height=600,  
        legend=dict(orientation='h', y=-0.3),
        title=dict(text=f'pred_count by gt_count (R²={r2:.4f})'),
        xaxis=dict(title=dict(text='gt_count')),
        yaxis=dict(title=dict(text='pred_count'), type='linear'),
    )
)

# 也可以加一个角落注释（可选）
fig.add_annotation(
    xref="paper", yref="paper",
    x=0.01, y=0.99,
    text=f"R² = {r2:.4f}",
    showarrow=False
)

fig.show()
